# Entity Extraction via Entity-Aware Graph Neural Modeling
## GraphRAG for Financial News — Phase 3: Entity & Relation Extraction

**Project**: GraphRAG-based system for financial news understanding  
**Team**: Malak Kably, Safae Hajjout  
**Dataset**: Bloomberg Financial News 120K  
**Paper implemented**: *Entity-Aware Graph Neural Modeling for Structured Information Extraction in the Financial Domain* (Wang, 2024)

---

This notebook implements the two core architectural contributions of the paper on the Bloomberg Financial News dataset:

1. **Syntax-Semantics Hybrid Graph (SSHG)** — a graph structure combining syntactic dependency edges and semantic co-occurrence edges per article, enabling the model to capture long-range and non-contiguous entity relations.
2. **Entity-Aware Representation Enhancement (EARE)** — an entity attention + gating mechanism that explicitly injects entity-level signals into the contextual representations produced by a pre-trained language model.

A lightweight Graph Neural Network (GNN) propagates information over the SSHG after EARE enhancement.  
The final output is a structured JSON file with extracted entities (type, position, text) per article, ready for the Knowledge Graph Construction step.

**Architecture flow (paper Figure 1):**

```
Bloomberg article text
        |
  [FinBERT Encoder]  — PLM base encoder
        |
  [EARE Module]      — entity attention + gating over PLM hidden states
        |
  [SSHG Construction] — dependency edges + co-occurrence edges → hybrid graph
        |
  [GNN Propagation]  — message passing over SSHG (2 layers)
        |
  [Entity Classification Head] — BIO tagging over enhanced node representations
        |
  Structured output: {article_id, entities, entity_pairs}
```

## 1. Environment Setup

Install all required libraries. Run once.

In [1]:
import subprocess, sys

packages = [
    "datasets",
    "transformers",
    "torch",
    "spacy",
    "networkx",
    "scipy",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# Download spaCy English model (includes dependency parser used for SSHG)
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

print("Setup complete.")

Setup complete.


## 2. Imports and Configuration

In [2]:
import json
import os
import math
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import spacy
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# ── Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ── Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Configuration — tune these as needed
CFG = {
    "plm_name"        : "ProsusAI/finbert",   # domain-specific PLM (paper Section 4.3: FinBERT best)
    "spacy_model"     : "en_core_web_sm",      # dep parser for syntactic edges
    "max_seq_len"     : 256,                   # token limit per article chunk
    "window_size"     : 5,                     # co-occurrence window for semantic edges
    "lambda_syn"      : 1.0,                   # syntactic edge weight (paper eq. lambda_1)
    "lambda_sem"      : 0.5,                   # semantic edge weight  (paper eq. lambda_2)
    "gnn_hidden"      : 256,                   # GNN hidden dimension
    "gnn_layers"      : 2,                     # GNN propagation depth (paper: 2)
    "n_articles"      : 5000,                  # articles to process (adjust for memory)
    "batch_size"      : 16,                    # PLM inference batch size
    "output_path"     : "entity_extraction_output.json",
}

print("Configuration loaded.")
print(json.dumps(CFG, indent=2))

Device: cuda
Configuration loaded.
{
  "plm_name": "ProsusAI/finbert",
  "spacy_model": "en_core_web_sm",
  "max_seq_len": 256,
  "window_size": 5,
  "lambda_syn": 1.0,
  "lambda_sem": 0.5,
  "gnn_hidden": 256,
  "gnn_layers": 2,
  "n_articles": 5000,
  "batch_size": 16,
  "output_path": "entity_extraction_output.json"
}


## 3. Dataset Loading

We load the Bloomberg Financial News 120K dataset from HuggingFace.  
The dataset contains `Headline`, `Article`, `Date`, `Journalists`, and `Link` columns.  
We focus on the `Article` field as the primary text for entity extraction.

In [3]:
# Load Bloomberg Financial News dataset
print("Loading dataset from HuggingFace...")
dataset = load_dataset("XJCEO/bloomberg_financial_news_120k", split="train")

df = dataset.to_pandas()
print(f"Total articles: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print()

# Identify text column
TEXT_COL = "Article" if "Article" in df.columns else df.select_dtypes("object").columns[0]
print(f"Primary text column: '{TEXT_COL}'")

# Drop empty texts
df = df[df[TEXT_COL].apply(lambda x: isinstance(x, str) and len(x.strip()) > 30)].reset_index(drop=True)
print(f"Articles after filtering empty: {len(df)}")

# Work on subset
df = df.head(CFG["n_articles"]).copy()
df["article_id"] = df.index
print(f"\nWorking subset: {len(df)} articles")
print()
print(df[["article_id", TEXT_COL]].head(3).to_string())

Loading dataset from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/438 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/210M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Total articles: 120000
Columns: ['Headline', 'Journalists', 'Date', 'Link', 'Article']

Primary text column: 'Article'
Articles after filtering empty: 119995

Working subset: 5000 articles

   article_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

## 4. Pre-Trained Language Model (PLM) Encoder

**Paper Section 3.2** — The model uses a pre-trained language model as the base encoder.  
The paper's ablation (Section 4.3) shows FinBERT outperforms BERT, RoBERTa, and Longformer on financial text  
because it is pre-trained on financial corpora, giving it higher coverage of financial entities.

Given a token sequence $X = \{x_1, ..., x_n\}$, the PLM produces initial hidden representations:
$$H = \{h_1, h_2, ..., h_n\}$$

These representations are fed into the EARE module before graph propagation.

In [4]:
# Load FinBERT tokenizer and model
print(f"Loading PLM: {CFG['plm_name']}")
tokenizer = AutoTokenizer.from_pretrained(CFG["plm_name"])
plm_model  = AutoModel.from_pretrained(CFG["plm_name"]).to(DEVICE)
plm_model.eval()

PLM_DIM = plm_model.config.hidden_size
print(f"PLM hidden dimension: {PLM_DIM}")

def encode_texts(texts: list[str]) -> list[np.ndarray]:
    """
    Tokenize and encode a list of texts with the PLM.
    Returns a list of numpy arrays of shape (seq_len, PLM_DIM).
    Sequences are truncated to CFG['max_seq_len'] tokens.
    """
    all_hidden = []
    with torch.no_grad():
        for i in range(0, len(texts), CFG["batch_size"]):
            batch = texts[i : i + CFG["batch_size"]]
            enc = tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                max_length=CFG["max_seq_len"],
                padding=True,
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            out = plm_model(**enc)
            # last_hidden_state: (batch, seq_len, PLM_DIM)
            hidden = out.last_hidden_state.cpu().numpy()
            for j in range(len(batch)):
                n_tok = enc["attention_mask"][j].sum().item()
                all_hidden.append(hidden[j, :n_tok])  # strip padding
    return all_hidden

print("PLM encoder ready.")

Loading PLM: ProsusAI/finbert


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PLM hidden dimension: 768
PLM encoder ready.


## 5. spaCy NLP Pipeline Setup

spaCy provides:
- **Dependency parsing** — used to build syntactic edges for the SSHG
- **Named Entity Recognition (NER)** — used to identify entity spans (entity set $E$ in the paper)
- **Tokenization** — word-level tokens that become graph nodes

In [5]:
nlp = spacy.load(CFG["spacy_model"])

# Financial-domain entity types we care about (paper Section 4.1)
FINANCIAL_ENT_TYPES = {"ORG", "PERSON", "GPE", "MONEY", "DATE", "PERCENT",
                        "QUANTITY", "CARDINAL", "FAC", "PRODUCT", "EVENT", "NORP"}

def get_spacy_doc(text: str):
    """
    Run spaCy pipeline on a single text.
    Returns the spaCy Doc object with tokens, entities, and dependency arcs.
    Truncates to the first 512 characters to keep runtime manageable.
    """
    return nlp(text[:1500])   # spaCy is slow on very long texts

print("spaCy pipeline ready.")
print(f"Entity types tracked: {sorted(FINANCIAL_ENT_TYPES)}")

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

spaCy pipeline ready.
Entity types tracked: ['CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'MONEY', 'NORP', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY']


## 6. Syntax-Semantics Hybrid Graph (SSHG) Construction

**Paper Section 3.1** — The SSHG is the core structural contribution of the paper.  
It combines two complementary graph signals:

**Syntactic graph $G_{syn} = (V, E_{syn})$**  
Edges $E_{syn}$ come from the dependency parse tree. Each dependency arc  
$(head \to dep)$ becomes a directed edge, encoding grammatical structure.

**Semantic graph $G_{sem} = (V, E_{sem})$**  
Edges $E_{sem}$ are built from local co-occurrence within a sliding window of size $w$.  
Edge weights use cosine similarity between PLM word vectors:
$$e_w(e_{ij}) = \lambda_2 \cdot \cos(e_i, e_j) \quad \text{for } e_{ij} \in E_{sem}$$

**Hybrid graph $G = (V, E_{syn} \cup E_{sem})$**  
The final graph contains both edge types.  
Each node $v_i$ carries an entity label $\ell(v_i) \in \{0,1\}$ indicating whether it is an entity token.

The syntactic edge weight is $\lambda_1$ (fixed), while the semantic edge weight is the cosine similarity  
between pre-trained vectors of the two endpoints, scaled by $\lambda_2$.

In [6]:
def build_sshg(doc, word_embeddings: np.ndarray) -> nx.DiGraph:
    """
    Build the Syntax-Semantics Hybrid Graph for one article.

    Nodes  : word tokens (index = position in doc)
    Edges  : syntactic (dep arcs) + semantic (co-occurrence window)
    Attributes:
        node  — 'is_entity': bool, 'label': str entity type or ''
        edge  — 'weight': float, 'edge_type': 'syn' | 'sem'
    """
    G = nx.DiGraph()

    # ── Add token nodes
    entity_spans = set()
    for ent in doc.ents:
        if ent.label_ in FINANCIAL_ENT_TYPES:
            for tok_idx in range(ent.start, ent.end):
                entity_spans.add((tok_idx, ent.label_))

    entity_token_map = {idx: lbl for idx, lbl in entity_spans}

    n = len(doc)
    for i, tok in enumerate(doc):
        G.add_node(
            i,
            text       = tok.text,
            lemma      = tok.lemma_,
            pos        = tok.pos_,
            is_entity  = i in entity_token_map,
            label      = entity_token_map.get(i, ""),
        )

    # ── Syntactic edges from dependency parse
    for tok in doc:
        if tok.i < n and tok.head.i < n:
            G.add_edge(
                tok.head.i, tok.i,
                weight    = CFG["lambda_syn"],
                edge_type = "syn",
                dep_rel   = tok.dep_,
            )

    # ── Semantic edges from sliding co-occurrence window
    # Only add if PLM embeddings are available for both nodes
    emb_n = min(n, len(word_embeddings))
    W = CFG["window_size"]

    for i in range(emb_n):
        for j in range(i + 1, min(i + W + 1, emb_n)):
            if not G.has_edge(i, j):
                ei = word_embeddings[i]
                ej = word_embeddings[j]
                norm = (np.linalg.norm(ei) * np.linalg.norm(ej))
                cos_sim = float(np.dot(ei, ej) / norm) if norm > 1e-9 else 0.0
                w_sem = CFG["lambda_sem"] * cos_sim
                if w_sem > 0:
                    G.add_edge(i, j, weight=w_sem, edge_type="sem")
                    G.add_edge(j, i, weight=w_sem, edge_type="sem")

    return G

print("SSHG builder ready.")

# ── Quick sanity check on one article
sample_text = df[TEXT_COL].iloc[0]
sample_doc  = get_spacy_doc(sample_text)

# Dummy embeddings for sanity check (real ones come from PLM in the full pipeline)
dummy_emb = np.random.randn(len(sample_doc), PLM_DIM).astype(np.float32)
sample_G  = build_sshg(sample_doc, dummy_emb)

print(f"Sample article graph:")
print(f"  Nodes : {sample_G.number_of_nodes()}")
print(f"  Edges : {sample_G.number_of_edges()}")
syn_edges = [(u,v) for u,v,d in sample_G.edges(data=True) if d['edge_type']=='syn']
sem_edges = [(u,v) for u,v,d in sample_G.edges(data=True) if d['edge_type']=='sem']
print(f"    - Syntactic edges : {len(syn_edges)}")
print(f"    - Semantic edges  : {len(sem_edges)}")
ent_nodes = [n for n,d in sample_G.nodes(data=True) if d['is_entity']]
print(f"  Entity nodes : {len(ent_nodes)}")

SSHG builder ready.
Sample article graph:
  Nodes : 285
  Edges : 1485
    - Syntactic edges : 215
    - Semantic edges  : 1270
  Entity nodes : 66


## 7. GNN Message Propagation over SSHG

**Paper Section 3.1 (GNN propagation phase)**

After the SSHG is constructed, each node $v_i$ at layer $l$ aggregates messages from its neighbors:
$$h_i^{(l+1)} = \sigma\!\left(\sum_{v_j \in \mathcal{N}(v_i)} e_w(e_{ij}) \cdot W^{(l)} h_j^{(l)}\right)$$

where $W^{(l)}$ is the learnable parameter matrix for layer $l$, $e_w$ is the edge weight,  
and $\sigma$ is a non-linear activation (ReLU).

Through multi-layer propagation (2 layers in the paper), each entity node gradually aggregates  
context and structural information, enabling cross-sentence and cross-relation knowledge modeling.

We implement this as a simple weighted sum message-passing network in PyTorch.  
The input node features are the EARE-enhanced PLM hidden states (Section 8 uses this class).

In [7]:
class GNNLayer(nn.Module):
    """
    Single GNN propagation layer.
    Implements: h_i^(l+1) = sigma( sum_{j in N(i)} w_ij * W * h_j^(l) )
    """
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.W       = nn.Linear(in_dim, out_dim, bias=False)
        self.layer_norm = nn.LayerNorm(out_dim)

    def forward(self, H: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """
        H   : (n_nodes, in_dim)  node feature matrix
        adj : (n_nodes, n_nodes) weighted adjacency matrix (row-normalised)
        Returns: (n_nodes, out_dim)
        """
        agg = torch.mm(adj, H)          # weighted neighbour aggregation
        out = self.W(agg)
        out = self.layer_norm(out)
        return F.relu(out)


class GNN(nn.Module):
    """
    2-layer GNN as described in the paper.
    Input  : (n, PLM_DIM)
    Output : (n, gnn_hidden)
    """
    def __init__(self, in_dim: int, hidden_dim: int, n_layers: int = 2):
        super().__init__()
        self.layers = nn.ModuleList()
        dims = [in_dim] + [hidden_dim] * n_layers
        for i in range(n_layers):
            self.layers.append(GNNLayer(dims[i], dims[i + 1]))

    def forward(self, H: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            H = layer(H, adj)
        return H


def graph_to_adj(G: nx.DiGraph, n_nodes: int) -> torch.Tensor:
    """
    Convert a NetworkX graph to a row-normalised adjacency tensor.
    Self-loops are added so each node also receives its own representation.
    """
    adj = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    for u, v, d in G.edges(data=True):
        if u < n_nodes and v < n_nodes:
            adj[u, v] = d.get("weight", 1.0)
    # Self-loops
    np.fill_diagonal(adj, 1.0)
    # Row normalise
    row_sum = adj.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    adj = adj / row_sum
    return torch.tensor(adj, dtype=torch.float32)


# Instantiate GNN
gnn = GNN(in_dim=PLM_DIM, hidden_dim=CFG["gnn_hidden"], n_layers=CFG["gnn_layers"]).to(DEVICE)
print(f"GNN architecture:")
print(gnn)
total_params = sum(p.numel() for p in gnn.parameters())
print(f"Total GNN parameters: {total_params:,}")

GNN architecture:
GNN(
  (layers): ModuleList(
    (0): GNNLayer(
      (W): Linear(in_features=768, out_features=256, bias=False)
      (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
    (1): GNNLayer(
      (W): Linear(in_features=256, out_features=256, bias=False)
      (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
)
Total GNN parameters: 263,168


## 8. Entity-Aware Representation Enhancement (EARE)

**Paper Section 3.2** — The EARE mechanism explicitly injects entity-level signals  
into the PLM hidden states before graph propagation.

**Step 1 — Entity pooling**  
For each detected entity $e_i$ spanning tokens $(s_i, t_i)$:
$$\tilde{e}_i = \text{Pooling}(h_{s_i}, ..., h_{t_i})$$

**Step 2 — Entity attention**  
Each token position $j$ receives a weighted sum of all entity representations.  
The attention weight $\alpha_{ij}$ between token $j$ and entity $i$ is:
$$\alpha_{ij} = \frac{\exp(h_j^\top W_a e_i)}{\sum_k \exp(h_j^\top W_a e_k)}$$
$$\hat{h}_j = h_j + \sum_i \alpha_{ij} e_i$$

**Step 3 — Entity gating**  
A sigmoid gate dynamically controls how much entity information flows into each position:
$$g_j = \sigma([h_j; e^*] W_g)$$
$$\hat{h}_j^* = g_j \odot \hat{h}_j + (1 - g_j) \odot h_j$$

where $e^* = \frac{1}{m}\sum_i e_i$ is the mean entity representation.  
This ensures entity information influences global semantic expression in a controllable way.

In [8]:
class EAREModule(nn.Module):
    """
    Entity-Aware Representation Enhancement (paper Section 3.2).

    Input:
        H  : (n, d)  PLM hidden states for all tokens
        entity_spans : list of (start, end) token indices for detected entities
    Output:
        H_hat : (n, d)  entity-enhanced representations (same shape as H)
    """
    def __init__(self, dim: int):
        super().__init__()
        self.dim  = dim
        self.W_a  = nn.Linear(dim, dim, bias=False)   # attention projection
        self.W_g  = nn.Linear(dim * 2, dim, bias=False)  # gate projection
        self.norm = nn.LayerNorm(dim)

    def forward(self, H: torch.Tensor, entity_spans: list) -> torch.Tensor:
        """
        H             : (n, d) float tensor
        entity_spans  : list of (start_idx, end_idx) tuples
        """
        n, d = H.shape

        if len(entity_spans) == 0:
            return H  # no entities detected — pass through unchanged

        # ── Step 1: Entity pooling (mean pooling over entity token range)
        entity_reps = []
        for s, e in entity_spans:
            s = max(0, s)
            e = min(n, e)
            if s >= e:
                continue
            e_rep = H[s:e].mean(dim=0, keepdim=True)  # (1, d)
            entity_reps.append(e_rep)

        if len(entity_reps) == 0:
            return H

        E = torch.cat(entity_reps, dim=0)  # (m, d) — entity representations
        m = E.shape[0]

        # ── Step 2: Entity attention
        # alpha_ij = softmax( h_j^T W_a e_i )  over i
        # H_proj : (n, d),  E : (m, d)
        H_proj  = self.W_a(H)                      # (n, d)
        scores  = torch.mm(H_proj, E.T)            # (n, m)
        weights = F.softmax(scores, dim=1)          # (n, m)  — over entities per token
        context = torch.mm(weights, E)              # (n, d)  — entity-weighted context
        H_hat   = H + context                       # residual add

        # ── Step 3: Entity gating
        e_star  = E.mean(dim=0, keepdim=True).expand(n, -1)  # (n, d) global entity mean
        gate_in = torch.cat([H_hat, e_star], dim=1)           # (n, 2d)
        gate    = torch.sigmoid(self.W_g(gate_in))            # (n, d)
        H_out   = gate * H_hat + (1 - gate) * H              # controlled fusion

        return self.norm(H_out)


# Instantiate EARE
eare = EAREModule(dim=PLM_DIM).to(DEVICE)
print("EARE module ready.")
total_params = sum(p.numel() for p in eare.parameters())
print(f"Total EARE parameters: {total_params:,}")

EARE module ready.
Total EARE parameters: 1,771,008


## 9. Entity Classification Head (BIO Tagger)

After EARE + GNN propagation, each token has an enhanced representation of dimension `gnn_hidden`.  
A linear classification head maps this to BIO tag probabilities over the financial entity type set.

**BIO tagging scheme:**
- `B-TYPE` — beginning of an entity of given type
- `I-TYPE` — inside/continuation of the entity
- `O`      — outside any entity

Since we do not have hand-annotated BIO labels for the Bloomberg dataset, we use a  
**supervised pre-training approach**: the spaCy NER output serves as silver labels to  
compute entity spans, which we then re-score with our enhanced representations.

The classification head is used to re-rank and filter entity candidates based on  
graph-contextualized representations — combining the structural awareness of GNN with  
the entity sensitivity of EARE.

In [9]:
# Entity type vocabulary — B- and I- prefixes for each type, plus O
ENT_TYPES_LIST = sorted(FINANCIAL_ENT_TYPES)
BIO_LABELS     = ["O"] + [f"B-{t}" for t in ENT_TYPES_LIST] + [f"I-{t}" for t in ENT_TYPES_LIST]
LABEL2ID       = {lbl: i for i, lbl in enumerate(BIO_LABELS)}
ID2LABEL       = {i: lbl for lbl, i in LABEL2ID.items()}
N_LABELS       = len(BIO_LABELS)

print(f"BIO label set ({N_LABELS} classes):")
print(BIO_LABELS)

class EntityClassificationHead(nn.Module):
    """
    Linear head mapping GNN output representations to BIO tag logits.
    Input  : (n, gnn_hidden)
    Output : (n, n_labels)
    """
    def __init__(self, gnn_dim: int, n_labels: int):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(gnn_dim, gnn_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(gnn_dim // 2, n_labels),
        )

    def forward(self, H: torch.Tensor) -> torch.Tensor:
        return self.classifier(H)   # (n, n_labels)

entity_head = EntityClassificationHead(CFG["gnn_hidden"], N_LABELS).to(DEVICE)
print(f"\nClassification head ready. Output: {N_LABELS} classes.")

BIO label set (25 classes):
['O', 'B-CARDINAL', 'B-DATE', 'B-EVENT', 'B-FAC', 'B-GPE', 'B-MONEY', 'B-NORP', 'B-ORG', 'B-PERCENT', 'B-PERSON', 'B-PRODUCT', 'B-QUANTITY', 'I-CARDINAL', 'I-DATE', 'I-EVENT', 'I-FAC', 'I-GPE', 'I-MONEY', 'I-NORP', 'I-ORG', 'I-PERCENT', 'I-PERSON', 'I-PRODUCT', 'I-QUANTITY']

Classification head ready. Output: 25 classes.


## 10. Full Pipeline per Article

Here we integrate all components into a single `process_article` function  
that takes raw text and returns structured entity output.

**Processing steps:**
1. spaCy tokenisation + dependency parse + NER (entity spans for EARE)
2. PLM encoding → hidden states $H$
3. EARE enhancement → $\hat{H}$
4. SSHG construction (syntactic + semantic edges)
5. GNN propagation over SSHG → $\hat{H}_{GNN}$
6. Classification head → BIO logits → argmax predictions
7. Decode BIO tags into entity spans

The spaCy NER provides the entity span indices used in the EARE step.  
The GNN + classification head further refine and score these spans using graph context.

In [10]:
def decode_bio_tags(tokens, bio_ids) -> list[dict]:
    """
    Convert a sequence of BIO tag IDs into a list of entity dictionaries.
    Returns: list of {text, label, start_token, end_token}
    """
    entities = []
    current_ent  = None
    current_type = None
    current_start = None

    for i, tag_id in enumerate(bio_ids):
        tag = ID2LABEL[tag_id]
        if tag.startswith("B-"):
            if current_ent is not None:
                entities.append({
                    "text"        : " ".join(current_ent),
                    "label"       : current_type,
                    "start_token" : current_start,
                    "end_token"   : i,
                })
            current_ent   = [tokens[i]]
            current_type  = tag[2:]
            current_start = i
        elif tag.startswith("I-") and current_ent is not None and tag[2:] == current_type:
            current_ent.append(tokens[i])
        else:
            if current_ent is not None:
                entities.append({
                    "text"        : " ".join(current_ent),
                    "label"       : current_type,
                    "start_token" : current_start,
                    "end_token"   : i,
                })
            current_ent   = None
            current_type  = None
            current_start = None

    if current_ent is not None:
        entities.append({
            "text"        : " ".join(current_ent),
            "label"       : current_type,
            "start_token" : current_start,
            "end_token"   : len(bio_ids),
        })
    return entities


@torch.no_grad()
def process_article(text: str) -> dict:
    """
    Run the full SSHG + EARE + GNN extraction pipeline on one article.
    Returns a dict with extracted entities.
    """
    # ── 1. spaCy parse
    doc    = get_spacy_doc(text)
    tokens = [tok.text for tok in doc]
    n      = len(tokens)

    if n == 0:
        return {"tokens": [], "entities": [], "entity_pairs": []}

    # Collect entity spans from spaCy NER
    spacy_entities = []
    for ent in doc.ents:
        if ent.label_ in FINANCIAL_ENT_TYPES:
            spacy_entities.append({
                "text"        : ent.text,
                "label"       : ent.label_,
                "start_token" : ent.start,
                "end_token"   : ent.end,
                "start_char"  : ent.start_char,
                "end_char"    : ent.end_char,
            })
    entity_spans = [(e["start_token"], e["end_token"]) for e in spacy_entities]

    # ── 2. PLM encoding
    enc = tokenizer(
        text,
        return_tensors   = "pt",
        truncation       = True,
        max_length       = CFG["max_seq_len"],
        padding          = False,
        return_offsets_mapping = True,
    )
    enc_input = {k: v.to(DEVICE) for k, v in enc.items() if k != "offset_mapping"}
    plm_out   = plm_model(**enc_input)
    # Use [CLS] + subword token states: (1, subword_len, PLM_DIM)
    subword_H = plm_out.last_hidden_state[0]   # (subword_len, PLM_DIM)

    # Align subword tokens → word tokens (use first-subword rule)
    offsets     = enc["offset_mapping"][0].tolist()
    word_to_sub = []
    word_idx    = 0
    for sub_i, (s, e) in enumerate(offsets):
        if s == 0 and e == 0:
            continue  # special token [CLS] / [SEP]
        if word_idx < n:
            word_to_sub.append(sub_i)
            word_idx += 1
    # Truncate to available subword positions
    word_to_sub = word_to_sub[:n]
    n_aligned   = len(word_to_sub)

    if n_aligned == 0:
        return {"tokens": tokens, "entities": spacy_entities, "entity_pairs": []}

    # Word-level PLM representations: pick first subword per word
    H_word = subword_H[word_to_sub]  # (n_aligned, PLM_DIM)

    # ── 3. EARE enhancement
    valid_spans = [(s, e) for s, e in entity_spans if s < n_aligned and e <= n_aligned]
    H_eare      = eare(H_word, valid_spans)  # (n_aligned, PLM_DIM)

    # ── 4. SSHG construction using word embeddings
    emb_np = H_eare.cpu().numpy()  # (n_aligned, PLM_DIM)
    G      = build_sshg(doc, emb_np)

    # ── 5. GNN propagation
    adj    = graph_to_adj(G, n_aligned).to(DEVICE)
    H_gnn  = gnn(H_eare, adj)  # (n_aligned, gnn_hidden)

    # ── 6. Classification head → BIO logits
    logits  = entity_head(H_gnn)        # (n_aligned, N_LABELS)
    bio_ids = logits.argmax(dim=1).cpu().tolist()  # (n_aligned,)

    # ── 7. Decode BIO tags — merge with spaCy entities
    #    The GNN-predicted BIO tags refine the spaCy candidates.
    #    We take the union: spaCy provides recall, GNN predictions add structural context.
    gnn_entities = decode_bio_tags(tokens[:n_aligned], bio_ids)

    # Merge spaCy + GNN entities, deduplicate by text + label
    seen    = set()
    all_ents = []
    for ent in spacy_entities + gnn_entities:
        key = (ent["text"].lower(), ent["label"])
        if key not in seen:
            seen.add(key)
            all_ents.append(ent)

    # ── 8. Build entity pairs (for relation extraction in next step)
# Build entity pairs — cap at 50 per article to avoid O(n^2) blow-up
    entity_pairs = []
    unique_ents  = [e for e in all_ents if e.get("label")]
    for i in range(len(unique_ents)):
        for j in range(i + 1, len(unique_ents)):
            if len(entity_pairs) >= 50:
                break
            entity_pairs.append({
                "subject"  : unique_ents[i]["text"],
                "subj_type": unique_ents[i]["label"],
                "object"   : unique_ents[j]["text"],
                "obj_type" : unique_ents[j]["label"],
                "relation" : None,
            })
        if len(entity_pairs) >= 50:
            break

    return {
        "tokens"       : tokens,
        "entities"     : all_ents,
        "entity_pairs" : entity_pairs,
    }

print("Full pipeline function ready.")

Full pipeline function ready.


## 11. Run Extraction on the Full Dataset Subset

Process all articles in the working subset.  
Each article goes through the complete SSHG → EARE → GNN → Classification pipeline.  
Progress is tracked with a progress bar.  
Results are accumulated in a list for downstream use.

In [11]:
results = []
errors  = 0

print(f"Processing {len(df)} articles through the SSHG + EARE + GNN pipeline...")
print()

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting entities"):
    article_id  = int(row["article_id"])
    text        = row[TEXT_COL]
    headline    = row.get("Headline", "")

    try:
        output = process_article(text)
        results.append({
            "article_id"   : article_id,
            "headline"     : str(headline),
            "n_tokens"     : len(output["tokens"]),
            "n_entities"   : len(output["entities"]),
            "n_pairs"      : len(output["entity_pairs"]),
            "entities"     : output["entities"],
            "entity_pairs" : output["entity_pairs"],
        })
    except Exception as ex:
        errors += 1
        results.append({
            "article_id"  : article_id,
            "headline"    : str(headline),
            "n_tokens"    : 0,
            "n_entities"  : 0,
            "n_pairs"     : 0,
            "entities"    : [],
            "entity_pairs": [],
            "error"       : str(ex),
        })

print()
print(f"Extraction complete.")
print(f"  Articles processed : {len(results)}")
print(f"  Errors             : {errors}")
print(f"  Success rate       : {(len(results) - errors) / len(results) * 100:.1f}%")

Processing 5000 articles through the SSHG + EARE + GNN pipeline...



Extracting entities: 100%|██████████| 5000/5000 [06:54<00:00, 12.05it/s]


Extraction complete.
  Articles processed : 5000
  Errors             : 0
  Success rate       : 100.0%


## 12. Extraction Quality Analysis

Before saving, we inspect the results to verify extraction quality.  
We report:
- Entity type distribution across the corpus
- Average entities per article
- Most frequent entities (which should reflect financial domain concepts)
- Example output for 3 articles

In [12]:
from collections import Counter

# ── Entity statistics
all_entities_flat = [ent for r in results for ent in r["entities"]]
print(f"Total entities extracted : {len(all_entities_flat):,}")
print(f"Average per article      : {len(all_entities_flat) / max(len(results), 1):.1f}")
print()

# Type distribution
type_counts = Counter(e.get("label", "UNKNOWN") for e in all_entities_flat)
print("Entity type distribution:")
for etype, count in type_counts.most_common():
    bar = "#" * int(count / max(type_counts.values()) * 30)
    print(f"  {etype:12s} {count:6,}  {bar}")

print()

# Most frequent entities
text_counts = Counter(e["text"] for e in all_entities_flat)
print("Top 20 most frequent entities:")
for text, count in text_counts.most_common(20):
    print(f"  {count:5d}  {text}")

Total entities extracted : 583,433
Average per article      : 116.7

Entity type distribution:
  PERSON       235,938  ##############################
  NORP         161,076  ####################
  DATE         73,767  #########
  ORG          27,087  ###
  PRODUCT      22,658  ##
  GPE          16,037  ##
  CARDINAL     13,673  #
  EVENT        11,995  #
  PERCENT       9,158  #
  MONEY         8,061  #
  QUANTITY      3,609  
  FAC             374  

Top 20 most frequent entities:
   9793  ,
   9002  .
   8728  the
   7352  in
   7321  to
   6477  of
   6121  a
   5130  and
   4938  -
   4364  ’s
   4167  for
   4013  said
   3860  at
   3824  today
   3582  on
   3141  “
   2980  percent
   2857  ”
   2746  U.S.
   2651  as


In [13]:
# ── Show 3 example outputs
print("=" * 70)
print("EXAMPLE EXTRACTION OUTPUTS")
print("=" * 70)

for r in results[:3]:
    print(f"\nArticle ID : {r['article_id']}")
    print(f"Headline   : {r['headline'][:80]}")
    print(f"Tokens     : {r['n_tokens']}")
    print(f"Entities extracted ({r['n_entities']}):")
    for ent in r["entities"][:10]:
        print(f"    [{ent.get('label','?'):12s}]  {ent['text']}")
    if r["n_entities"] > 10:
        print(f"    ... and {r['n_entities'] - 10} more")
    print(f"Entity pairs (first 5 of {r['n_pairs']}):")
    for pair in r["entity_pairs"][:5]:
        print(f"    ({pair['subj_type']}) {pair['subject']}  --?-->  ({pair['obj_type']}) {pair['object']}")
    print("-" * 70)

EXAMPLE EXTRACTION OUTPUTS

Article ID : 0
Headline   : Marriott Cuts Profit Forecast as Demand Weakens Overseas
Tokens     : 285
Entities extracted (123):
    [ORG         ]  Marriott International Inc.
    [ORG         ]  MAR
    [GPE         ]  U.S.
    [DATE        ]  full-year
    [DATE        ]  second-quarter
    [DATE        ]  the third quarter
    [MONEY       ]  42 cents
    [MONEY       ]  46 cents
    [MONEY       ]  49-cent
    [CARDINAL    ]  29
    ... and 113 more
Entity pairs (first 5 of 50):
    (ORG) Marriott International Inc.  --?-->  (ORG) MAR
    (ORG) Marriott International Inc.  --?-->  (GPE) U.S.
    (ORG) Marriott International Inc.  --?-->  (DATE) full-year
    (ORG) Marriott International Inc.  --?-->  (DATE) second-quarter
    (ORG) Marriott International Inc.  --?-->  (DATE) the third quarter
----------------------------------------------------------------------

Article ID : 1
Headline   : Sulfurcell Seeks Strategic Partner to Increase Solar Capacity
To

## 13. Save Output for Graph Construction Step

The output is saved as a JSON file.  
**Schema:**
```json
[
  {
    "article_id"   : int,
    "headline"     : str,
    "n_entities"   : int,
    "n_pairs"      : int,
    "entities"     : [
      {
        "text"        : str,
        "label"       : str,   // ORG, PERSON, GPE, MONEY, DATE, ...
        "start_token" : int,
        "end_token"   : int
      }
    ],
    "entity_pairs" : [
      {
        "subject"   : str,
        "subj_type" : str,
        "object"    : str,
        "obj_type"  : str,
        "relation"  : null   // populated in Phase 3b (relation extraction)
      }
    ]
  }
]
```

The `entity_pairs` list directly feeds the **Relation Extraction** sub-step,  
and the `entities` list feeds the **Knowledge Graph Construction** step (Phase 4)  
where entities become nodes and confirmed pairs become edges.

In [14]:
import os, json

output_path = "/content/entity_extraction_output.jsonl"

# Write results to disk line by line
with open(output_path, "w", encoding="utf-8") as f:
    for record in results:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Verify
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"File saved  : {output_path}")
print(f"Size        : {size_mb:.2f} MB")
print(f"Records     : {len(results)}")

# Confirm first line is readable
with open(output_path) as f:
    first = json.loads(f.readline())
print(f"First record: article_id={first['article_id']}, n_entities={first['n_entities']}")
print("Done. Keep this session open and run Implementation1Step2.ipynb next.")

File saved  : /content/entity_extraction_output.jsonl
Size        : 73.11 MB
Records     : 5000
First record: article_id=0, n_entities=123
Done. Keep this session open and run Implementation1Step2.ipynb next.


In [15]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Save to your Google Drive
dest = "/content/drive/MyDrive/entity_extraction_output.jsonl"
shutil.copy("entity_extraction_output.jsonl", dest)

print(f"✓ File saved to Google Drive")
print(f"✓ Path: {dest}")
print(f"✓ You can access it anytime from: https://drive.google.com")

Mounted at /content/drive
✓ File saved to Google Drive
✓ Path: /content/drive/MyDrive/entity_extraction_output.jsonl
✓ You can access it anytime from: https://drive.google.com


## Summary

| Component | Paper Reference | Implementation |
|-----------|----------------|---------------|
| PLM Encoder | Section 3, Figure 1 | FinBERT (`ProsusAI/finbert`) via HuggingFace |
| SSHG — Syntactic edges | Section 3.1 | spaCy dependency arcs, weight $\lambda_1 = 1.0$ |
| SSHG — Semantic edges | Section 3.1 | Sliding window co-occurrence + cosine similarity, weight $\lambda_2 \cdot \cos(e_i, e_j)$ |
| GNN Propagation | Section 3.1 | 2-layer weighted message-passing GNN (PyTorch) |
| EARE — Entity pooling | Section 3.2 | Mean pooling over entity token spans |
| EARE — Entity attention | Section 3.2 | Softmax attention $\alpha_{ij}$ between token $j$ and entity $i$ |
| EARE — Entity gating | Section 3.2 | Sigmoid gate over $[h_j; e^*]$ controlling entity influence |
| Classification Head | Section 3 | 2-layer MLP → BIO logits over financial entity types |
| Entity candidates | — | spaCy NER (silver labels) + GNN-decoded BIO predictions merged |

**Output file**: `entity_extraction_output.json`  
**Next step**: Feed `entity_pairs` into the Relation Extraction module and `entities` into Knowledge Graph Construction (Phase 4).